# OCCAM Python Analysis Notebook

This notebook demonstrates the complete OCCAM workflow using the Python bindings.
Following the OCCAM manual v3.4.1 workflow for reconstructability analysis.

## Workflow Overview
1. **Initialize** - Load data and configure OCCAM
2. **Search** - Find candidate models using various search algorithms
3. **Select** - Choose best model based on BIC, AIC, or Information criteria
4. **Fit** - Perform detailed analysis of the selected model
5. **Evaluate** - Assess model performance and generate reports

## 1. Setup and Configuration

In [11]:
import pyoccam
import pandas as pd
import numpy as np
from datetime import datetime
import os

# Configuration - Modify these for your analysis
DATA_FILE = "dementia05.txt"  # Your data file
SEARCH_TYPE = "loopless-up"   # Search algorithm: loopless-up, full-up, disjoint-up, chain-up
SEARCH_LEVELS = 7              # How deep to search (lattice levels)
SEARCH_WIDTH = 3               # How many models to keep at each level
REFERENCE_MODEL = "bottom"    # Reference for statistics: bottom or top
SELECTION_CRITERION = "bic"   # Model selection: bic, aic, or information

print(f"OCCAM Python v{pyoccam.__version__}")
print(f"Analysis started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Data file: {DATA_FILE}")

OCCAM Python v3.4.0
Analysis started: 2025-08-09 01:22:02
Data file: dementia05.txt


## 2. Initialize OCCAM Manager

In [12]:
# Initialize the OCCAM manager
manager = pyoccam.VBMManager()

# Load data file
args = ["occam", DATA_FILE]
success = manager.init_from_command_line(args)

if not success:
    raise RuntimeError(f"Failed to initialize OCCAM with {DATA_FILE}")

print("✓ OCCAM Manager initialized successfully")

# Display basic statistics
print("\nData Statistics:")
print("-" * 40)
print(manager.get_basic_statistics())

# Display variables
variables = manager.get_variable_list()
print(f"\nVariables ({len(variables)}):")
for i, var in enumerate(variables, 1):
    print(f"  {i:2d}. {var}")

print(f"\nSample size: {manager.get_sample_size()}")
print(f"Has test data: {manager.has_test_data()}")

✓ OCCAM Manager initialized successfully

Data Statistics:
----------------------------------------
State Space Size: 1.45119e+09
Sample Size: 424
H(data): 8.72792


Variables (19):
   1. APOE
   2. Gender
   3. Education
   4. AgeLastExam
   5. rs1801133
   6. rs3818361
   7. rs7561528
   8. rs744373
   9. rs6943822
  10. rs4298437
  11. rs7012010
  12. rs11136000
  13. rs10786998
  14. rs11193130
  15. rs610932
  16. rs3851179
  17. rs3764650
  18. rs3865444
  19. CaseControl

Sample size: 424
Has test data: False


## 3. Configure Report Format

In [13]:
# Configure report format
manager.set_report_separator(pyoccam.SPACESEP)  # Use space-separated format

# Configure which columns to display in reports
# According to manual: ID, Model, Level, H, dDF, dLR, Alpha, Inf, %dH(DV), dAIC, dBIC
report_columns = "ID$I, Model, Level$I, h, ddf, dLR, Alpha, Inf, %dH(DV), dAIC, dBIC"
manager.set_report_variables(report_columns)

# Set reference model for statistics
manager.set_ref_model(REFERENCE_MODEL)

print(f"Report format configured:")
print(f"  Separator: Space-separated")
print(f"  Reference: {REFERENCE_MODEL}")
print(f"  Columns: {report_columns}")

Report format configured:
  Separator: Space-separated
  Reference: bottom
  Columns: ID$I, Model, Level$I, h, ddf, dLR, Alpha, Inf, %dH(DV), dAIC, dBIC


## 4. Perform Model Search

Search for candidate models using the specified algorithm.
The search explores the model lattice to find models that balance goodness-of-fit with complexity.

In [14]:
print(f"Performing {SEARCH_TYPE} search...")
print(f"  Levels: {SEARCH_LEVELS}")
print(f"  Width: {SEARCH_WIDTH}")
print("-" * 60)

# Generate search report
search_report = manager.generate_search_report(
    search_type=SEARCH_TYPE,
    levels=SEARCH_LEVELS,
    width=SEARCH_WIDTH,
    include_test_data=False
)

# Display search results
print(search_report)

# Save search report to file
search_filename = f"search_report_{SEARCH_TYPE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
with open(search_filename, 'w') as f:
    f.write(search_report)
print(f"\n✓ Search report saved to {search_filename}")

Performing loopless-up search...
  Levels: 7
  Width: 3
------------------------------------------------------------
OCCAM version 3.4.0

Option settings:
Search direction up
Search width     3
Search levels    7
Search type      loopless-up
Report sort by   information
Report preference descending

State Space Size      1.451188e+09
Sample Size           424.000000
DV                    Z

Searching level:
1 : ... models evaluated ...
2 : ... models evaluated ...
3 : ... models evaluated ...
4 : ... models evaluated ...
5 : ... models evaluated ...
6 : ... models evaluated ...
7 : ... models evaluated ...

  ID   MODEL                     ID          Model          Level              H            dDF            dLR          Alpha            Inf        %dH(DV)           dAIC           dBIC
  20   IV:EdBGJKNPZ                                                       9.3538           9215                        1.0000                                                            
  20   IV:EdB

## 5. Extract Best Models

According to the OCCAM manual, the best models are summarized at the end of the search output.
These include models with the best (highest) values of dBIC and dAIC.

In [15]:
# Parse search report to find best models
# In a future version, these should be available directly from the manager
# For now, we'll extract from the report text

def extract_best_models(report_text):
    """Extract best model names from search report."""
    best_models = {}
    lines = report_text.split('\n')
    
    for i, line in enumerate(lines):
        if "Best Model(s) by dBIC:" in line:
            # Next line(s) contain the best BIC model(s)
            j = i + 1
            while j < len(lines) and lines[j].strip() and not "Best Model" in lines[j]:
                # Extract model name from line like: "16*  IV:ApZ:EdZ:CZ:KZ  BIC=45.7461"
                parts = lines[j].split()
                if len(parts) >= 2:
                    model_name = parts[1]
                    if not 'bic' in best_models:
                        best_models['bic'] = model_name
                j += 1
                
        elif "Best Model(s) by dAIC:" in line:
            j = i + 1
            while j < len(lines) and lines[j].strip() and not "Best Model" in lines[j]:
                parts = lines[j].split()
                if len(parts) >= 2:
                    model_name = parts[1]
                    if not 'aic' in best_models:
                        best_models['aic'] = model_name
                j += 1
                
        elif "Best Model(s) by Information" in line:
            j = i + 1
            while j < len(lines) and lines[j].strip() and not "Best Model" in lines[j]:
                parts = lines[j].split()
                if len(parts) >= 2:
                    model_name = parts[1]
                    if not 'information' in best_models:
                        best_models['information'] = model_name
                j += 1
    
    return best_models

# Extract best models
best_models = extract_best_models(search_report)

print("Best Models Found:")
print("=" * 40)
for criterion, model_name in best_models.items():
    print(f"  By {criterion.upper():12s}: {model_name}")

# Select model based on user preference
selected_model = best_models.get(SELECTION_CRITERION.lower())
if not selected_model:
    # Fallback to first available best model
    selected_model = list(best_models.values())[0] if best_models else None

if selected_model:
    print(f"\n✓ Selected model (by {SELECTION_CRITERION}): {selected_model}")
else:
    print("\n⚠ No best model found in search results")

Best Models Found:
  By BIC         : IV:Z
  By AIC         : IV:Z

✓ Selected model (by bic): IV:Z


## 6. Fit Selected Model

Perform detailed analysis of the selected model, including:
- Model structure and components
- Degrees of freedom and entropy
- Information captured
- Residuals and fit statistics

In [ ]:
if selected_model:
    print(f"Generating fit report for: {selected_model}")
    print("=" * 60)
    
    # Generate fit report
    fit_report = manager.generate_fit_report(selected_model)
    
    # Display fit report
    print(fit_report)
    
    # Save fit report to file
    fit_filename = f"fit_report_{selected_model.replace(':', '_')}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    with open(fit_filename, 'w') as f:
        f.write(fit_report)
    print(f"\n✓ Fit report saved to {fit_filename}")
else:
    print("No model selected for fitting.")

## 7. Model Statistics Summary

Get detailed statistics for the selected model.

In [ ]:
if selected_model:
    # Get model statistics
    stats = manager.get_model_statistics(selected_model)
    
    print(f"Model Statistics for {selected_model}:")
    print("=" * 60)
    
    # Create a formatted table of statistics
    stats_df = pd.DataFrame([
        ['Entropy (H)', stats.get('h', 'N/A'), 'bits'],
        ['Information', stats.get('information', 'N/A') * 100, '%'],
        ['AIC', stats.get('aic', 'N/A'), ''],
        ['BIC', stats.get('bic', 'N/A'), ''],
        ['Alpha', stats.get('alpha', 'N/A'), 'p-value'],
        ['LR', stats.get('lr', 'N/A'), 'chi-square'],
        ['% Correct', stats.get('pct_correct_data', 'N/A'), '%'],
        ['Incr. Alpha', stats.get('incr_alpha', 'N/A'), 'p-value']
    ], columns=['Statistic', 'Value', 'Unit'])
    
    print(stats_df.to_string(index=False))
    
    # Interpretation
    print("\nInterpretation:")
    print("-" * 40)
    
    if stats.get('alpha', 1.0) < 0.05:
        print("✓ Model is statistically significant (α < 0.05)")
    else:
        print("⚠ Model is not statistically significant (α ≥ 0.05)")
    
    info_pct = stats.get('information', 0) * 100
    print(f"✓ Model captures {info_pct:.1f}% of the information in the data")
    
    pct_correct = stats.get('pct_correct_data', 0)
    if pct_correct > 0:
        print(f"✓ Model correctly predicts {pct_correct:.1f}% of cases")

## 8. Compare Search Algorithms (Optional)

Compare different search algorithms to see which finds better models.

In [ ]:
# Optional: Compare different search algorithms
COMPARE_ALGORITHMS = False  # Set to True to enable comparison

if COMPARE_ALGORITHMS:
    algorithms = ["loopless-up", "full-up", "disjoint-up"]
    comparison_results = []
    
    print("Comparing Search Algorithms")
    print("=" * 60)
    
    for algo in algorithms:
        print(f"\nTesting {algo}...")
        
        # Run search
        report = manager.generate_search_report(
            search_type=algo,
            levels=3,  # Reduced for comparison
            width=3,
            include_test_data=False
        )
        
        # Extract best models
        best = extract_best_models(report)
        
        # Get statistics for best BIC model
        if 'bic' in best:
            stats = manager.get_model_statistics(best['bic'])
            comparison_results.append({
                'Algorithm': algo,
                'Best Model': best['bic'],
                'BIC': stats.get('bic', 'N/A'),
                'Information': stats.get('information', 0) * 100,
                '% Correct': stats.get('pct_correct_data', 'N/A')
            })
    
    # Display comparison
    if comparison_results:
        comparison_df = pd.DataFrame(comparison_results)
        print("\nAlgorithm Comparison Results:")
        print(comparison_df.to_string(index=False))
else:
    print("Algorithm comparison skipped (set COMPARE_ALGORITHMS=True to enable)")

## 9. Export Results

Create a summary of the analysis for reporting.

In [ ]:
# Create analysis summary
summary = {
    'Analysis Date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'Data File': DATA_FILE,
    'Sample Size': manager.get_sample_size(),
    'Number of Variables': len(variables),
    'Search Algorithm': SEARCH_TYPE,
    'Search Levels': SEARCH_LEVELS,
    'Search Width': SEARCH_WIDTH,
    'Selection Criterion': SELECTION_CRITERION,
    'Selected Model': selected_model if selected_model else 'None',
    'Search Report': search_filename,
    'Fit Report': fit_filename if selected_model else 'N/A'
}

# Save summary
summary_filename = f"analysis_summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
with open(summary_filename, 'w') as f:
    f.write("OCCAM Analysis Summary\n")
    f.write("=" * 60 + "\n\n")
    for key, value in summary.items():
        f.write(f"{key:25s}: {value}\n")

print("Analysis Summary:")
print("=" * 60)
for key, value in summary.items():
    print(f"{key:25s}: {value}")

print(f"\n✓ Summary saved to {summary_filename}")
print("\n✓ Analysis complete!")

## 10. Clean Up and Next Steps

In [ ]:
print("Next Steps:")
print("=" * 60)
print("1. Review the search report to understand the model space")
print("2. Examine the fit report for detailed model statistics")
print("3. Consider running with different search parameters:")
print("   - Increase SEARCH_LEVELS for deeper search")
print("   - Increase SEARCH_WIDTH to keep more models")
print("   - Try different SEARCH_TYPE algorithms")
print("4. If you have test data, evaluate model performance")
print("5. Generate hypergraph visualizations of the model structure")
print("\nRefer to the OCCAM manual for detailed interpretation of results.")